# EXP-07 v2 — Comparação exploratória de quatro classificadores e três conjuntos de características

**Objetivo:** comparar **4 modelos × 3 conjuntos de características = 12 combinações** sob divisões idênticas. Desenho congelado antes da avaliação: A = cinco atributos originais; B = retirar `n_tx`; C = apenas três entropias.

**Alerta científico:** a matriz é produto de seleção prévia de exemplos e pode ter apenas **um único contrato positivo**, tornando impossível uma validação confirmatória de generalização para protocolos/contratos novos. Caso não existam positivos de múltiplos contratos, o notebook registra métricas **somente exploratórias de validação por linhas**, nunca afirma desempenho em contratos desconhecidos. Se não for possível separar adequadamente, a execução interrompe ou identifica a limitação, sem criar rótulos artificiais.

**Dados:** uma matriz original congelada. Não usar alertas de 2026 ou pools de Privacy Pools como rótulos positivos/negativos. Modelos e hiperparâmetros fixos, sem busca de hiperparâmetros e sem escolher retrospectivamente a configuração vencedora. `recipient`, `dia` e `label` não são preditores. A seleção de conjuntos de características NÃO aumenta o tamanho do dataset.

In [ ]:
%pip -q install pyarrow scikit-learn pandas google-api-python-client


## 1. Baixar a matriz do Drive oficial
Usa a sessão autenticada da sua conta Google. **Não cole tokens nem URLs privadas de RPC.**

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from datetime import datetime, timezone
import io, os, hashlib, json

auth.authenticate_user()
service=build('drive','v3')
MATRIX_ID='1jUHAq6H9FKSgptv4nsCoOmFILH4Py_2S'
ROOT=Path('/content/EXP-07')
DEST=ROOT/('EXP-07_comparativo_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
DEST.mkdir(parents=True,exist_ok=False)
req=service.files().get_media(fileId=MATRIX_ID)
buf=io.BytesIO();downloader=MediaIoBaseDownload(buf,req)
done=False
while not done:
    _,done=downloader.next_chunk()
MATRIX=DEST/'features_treino_original.parquet';MATRIX.write_bytes(buf.getvalue())
H=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
print('Matriz bytes:',MATRIX.stat().st_size,'SHA256:',H(MATRIX),'Saída:',DEST)


## 2. Auditoria obrigatória da matriz

O número de **contratos positivos distintos** e a presença de classes conflitantes por endereço determinam a estratégia de validação. Amostras com o mesmo endereço em treino e teste podem inflar métricas. Valores não finitos são tratados dentro de cada *pipeline*; nenhuma transformação é ajustada com o conjunto de teste.

In [ ]:
import numpy as np, pandas as pd
D=pd.read_parquet(MATRIX)
FEATURES=['n_tx','value_entropy','fee_entropy','internal_value_entropy','call_count_mean']
FEATURE_SETS={
 'A_original_5':FEATURES,
 'B_sem_volume_4':[f for f in FEATURES if f!='n_tx'],
 'C_entropias_3':['value_entropy','fee_entropy','internal_value_entropy']
}
REQUIRED=['recipient','dia','label']+FEATURES
missing=[k for k in REQUIRED if k not in D.columns]
if missing:raise ValueError('Colunas obrigatórias ausentes: '+str(missing))
if D['label'].isna().any() or not D['label'].isin([0,1]).all():
    raise ValueError('Rótulos ausentes ou não binários')
D['recipient']=D['recipient'].astype(str).str.lower().str.strip()
D['dia']=pd.to_datetime(D['dia'],errors='coerce')
if D['dia'].isna().any() or not D['recipient'].str.match(r'^0x[0-9a-f]{40}$').all():
    raise ValueError('Datas/endereços inválidos')
D['label']=D['label'].astype(int)
for col in FEATURES:
    D[col]=pd.to_numeric(D[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
if set(D['label'])!={0,1}:raise ValueError('Uma das duas classes está ausente')
if any(D[col].notna().sum()==0 for col in FEATURES):
    raise ValueError('Ao menos uma característica é completamente vazia')
conflicts=(D.groupby('recipient')['label'].nunique()>1)
if conflicts.any():
    raise ValueError('Mesmo contrato com classes conflitantes: revisão manual necessária')
contract_counts=D.groupby('label')['recipient'].nunique().to_dict()
class_counts=D['label'].value_counts().sort_index().to_dict()
dup=int(D.duplicated(['recipient','dia']).sum())
report={
 'n_rows':int(len(D)),
 'n_columns':int(len(D.columns)),
 'class_counts':{str(k):int(v) for k,v in class_counts.items()},
 'contracts_by_class':{str(k):int(v) for k,v in contract_counts.items()},
 'distinct_contracts':int(D.recipient.nunique()),
 'duplicate_contract_day':dup,
 'days_by_class':{str(k):int(v) for k,v in D.groupby('label')['dia'].nunique().items()},
 'missing_numeric_by_feature':{f:int(D[f].isna().sum()) for f in FEATURES},
 'feature_sets':FEATURE_SETS,
 'matrix_sha256':H(MATRIX),
 'single_positive_contract':contract_counts.get(1,0)==1
}
(DEST/'auditoria_matriz.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(report,indent=2,ensure_ascii=False))
if contract_counts.get(1,0)==1:
 print('AVISO: apenas um contrato positivo; NÃO é possível validar generalização por contrato.')

## 3. Quatro classificadores × três configurações = 12 combinações

**Divisão única e reproduzível para todos os modelos/configurações.** Primeiro tentar dois *folds* estratificados **agrupados por `recipient`**, somente se houver ≥2 contratos positivos e ≥2 negativos e todas as dobras mantiverem ambas as classes. Caso contrário, usar `StratifiedKFold` de linhas **somente como diagnóstico exploratório**, explicitando o vazamento potencial e proibindo conclusões sobre novos contratos.

Todas as métricas são calculadas por *fold*, com mesmo limiar de classificação 0,5 e sementes fixas. `precision`, `recall`, `F1`, `balanced_accuracy` e matrizes de confusão descrevem **apenas** a validação adotada, não o desempenho em toda a rede Ethereum. O Random Forest é **retreinado** com os mesmos parâmetros da referência; não confundir com a execução do modelo antigo salvo em `joblib`.

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score,recall_score,f1_score,balanced_accuracy_score,confusion_matrix

seed=20261119
y=D['label'].astype(int)
groups=D['recipient']
models={
 'RandomForest':lambda: Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('clf',RandomForestClassifier(n_estimators=100,max_depth=5,random_state=seed,n_jobs=1))]),
 'RegressaoLogistica':lambda: Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler()),('clf',LogisticRegression(max_iter=2000,random_state=seed,class_weight='balanced'))]),
 'DecisionTree':lambda: Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('clf',DecisionTreeClassifier(max_depth=5,random_state=seed))]),
 'GradientBoosting':lambda: Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('clf',GradientBoostingClassifier(random_state=seed))])
}

def validate_splits(pairs,grouped):
    for tr,te in pairs:
        if set(y.iloc[tr])!={0,1} or set(y.iloc[te])!={0,1}:return False
        if grouped and set(groups.iloc[tr]) & set(groups.iloc[te]):return False
    return True

can_group=all(contract_counts.get(k,0)>=2 for k in (0,1))
splits=None
if can_group:
    try:
        candidate=list(StratifiedGroupKFold(n_splits=2,shuffle=True,random_state=seed).split(D,y,groups))
        if validate_splits(candidate,True): splits=candidate
    except ValueError:pass
if splits is not None:
    method='agrupada_por_contrato_2_folds_exploratoria'
    group_validation=True
else:
    minclass=int(y.value_counts().min())
    if minclass<2:raise ValueError('Não há exemplos suficientes para validação por linhas')
    folds=min(5,minclass)
    candidate=list(StratifiedKFold(n_splits=folds,shuffle=True,random_state=seed).split(D,y))
    if not validate_splits(candidate,False):raise ValueError('Não foi possível obter ambas as classes em cada fold')
    splits=candidate
    method='POR_LINHAS_EXPLORATORIA_SEM_VALIDACAO_POR_CONTRATO'
    group_validation=False

split_manifest=[]
for fold,(tr,te) in enumerate(splits,1):
    split_manifest.append({'fold':fold,'n_treino':int(len(tr)),'n_teste':int(len(te)),
       'n_positivos_treino':int(y.iloc[tr].sum()),'n_positivos_teste':int(y.iloc[te].sum()),
       'n_contratos_treino':int(groups.iloc[tr].nunique()),'n_contratos_teste':int(groups.iloc[te].nunique()),
       'contratos_compartilhados':int(len(set(groups.iloc[tr]) & set(groups.iloc[te]))),
       'index_treino':[int(i) for i in tr],'index_teste':[int(i) for i in te]})
(DEST/'folds_congelados.json').write_text(json.dumps({'method':method,'folds':split_manifest},indent=2),encoding='utf-8')
print('Método:',method,'folds:',len(splits),'contratos positivos:',contract_counts.get(1,0))

rows=[]
for feature_set,cols in FEATURE_SETS.items():
    X=D[cols]
    for model_name,make_model in models.items():
        for fold,(tr,te) in enumerate(splits,1):
            model=make_model()
            model.fit(X.iloc[tr],y.iloc[tr])
            pred=model.predict(X.iloc[te]);tn,fp,fn,tp=confusion_matrix(y.iloc[te],pred,labels=[0,1]).ravel()
            rows.append({'configuracao':feature_set,'n_atributos':len(cols),'modelo':model_name,
                'fold':fold,'n_treino':len(tr),'n_teste':len(te),'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
                'precision':float(precision_score(y.iloc[te],pred,zero_division=0)),
                'recall':float(recall_score(y.iloc[te],pred,zero_division=0)),
                'f1':float(f1_score(y.iloc[te],pred,zero_division=0)),
                'balanced_accuracy':float(balanced_accuracy_score(y.iloc[te],pred)),
                'metodo_validacao':method,'avaliacao_externa':False})
R=pd.DataFrame(rows)
R.to_csv(DEST/'metricas_12_combinacoes_por_fold.csv',index=False)
metrics=['precision','recall','f1','balanced_accuracy']
S=R.groupby(['configuracao','modelo'])[metrics].agg(['mean','std'])
S.columns=['_'.join(c) for c in S.columns]
S=S.reset_index().round(4)
S.to_csv(DEST/'resumo_12_combinacoes.csv',index=False)
print(S.to_string(index=False))
print('AVISO: métricas internas, não medem generalização a outros protocolos.')

## 4. Consolidação e controles interpretativos

O arquivo de saída terá 12 linhas de resumo, uma por combinação, além das métricas por dobra e um manifesto de integridade. Não selecionamos a melhor configuração com base em 2020/2026. Comparações A→B→C mostram **associação descritiva** do desempenho com a remoção de atributos; não são inferências causais sobre a importância real dessas características em contratos novos.

In [ ]:
manifest={'experimento':'EXP-07','versao':'v2_ablation','gerado_utc':datetime.now(timezone.utc).isoformat(),
 'dataset_matriz_id':MATRIX_ID,'metodo_validacao':method,'n_linhas':len(D),
 'contratos_positivos_distintos':int(contract_counts.get(1,0)),
 'contratos_negativos_distintos':int(contract_counts.get(0,0)),
 'validacao_agrupada_por_contrato':bool(group_validation),
 'generalizacao_para_outro_protocolo_demonstrada':False,
 'n_modelos':len(models),'n_configuracoes':len(FEATURE_SETS),'n_combinacoes':len(models)*len(FEATURE_SETS),
 'feature_sets':FEATURE_SETS,'semente':seed,
 'limites':['Amostras positivas podem vir de um único contrato',
 'Validação por linhas pode vazar identidade e tempo',
 'Amostras rotuladas pertencem ao conjunto original e não representam a população da Ethereum',
 '2020 e janeiro/2026 não usados como teste externo rotulado',
 'Ablação exploratória não comprova importância causal ou generalização a novos protocolos',
 'Não usar matriz de alertas como ground truth'],
 'arquivos_sha256':{p.name:H(p) for p in sorted(DEST.iterdir()) if p.is_file()}}
(DEST/'manifesto_exp07.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
parent='1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'
folder=service.files().create(body={'name':DEST.name,'mimeType':'application/vnd.google-apps.folder','parents':[parent]},fields='id,name').execute()
for p in sorted(DEST.iterdir()):
    if p.is_file():
        obj=service.files().create(body={'name':p.name,'parents':[folder['id']]},
           media_body=MediaFileUpload(str(p),mimetype='application/octet-stream',resumable=True),fields='id,name').execute()
        print('Salvo:',obj['name'])
print('RESULTADOS: https://drive.google.com/drive/folders/'+folder['id'])
print('VALIDAÇÃO:',method,'| 12 combinações; NÃO publicar como generalização para novos protocolos.')